In [1]:
# 测试
import torch
import cutlass
import cutlass.cute as cute
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
print(cutlass.__version__)

@cute.jit
def demo():
    layout = cute.make_layout((4,), stride=(1,))
    comp = cute.complement(layout, 32)
    print(layout, comp)

    layout_1 = cute.make_layout((1,32))
    print(layout_1)

    layout_2 = cute.make_layout((1,32),stride=(32,1))
    print(layout_2)

demo()

True NVIDIA GeForce RTX 5070 Ti
4.7.1
(4):(1) 8:4
(1,32):(0,1)
(1,32):(32,1)


In [2]:
# local_tile 练习：给定 1024 元素的 1D Tensor，BLOCK=128， 用 local_tile 切出所有 tile，打印每个 tile 的起始偏移。
import cutlass 
import cutlass.cute as cute   # 官方包是 cutlass，cute 是它的子模块
import torch
from cutlass.cute.runtime import from_dlpack
import cuda.bindings.driver as cuda 

n = 1024
block_size = 128

@cute.kernel
def local_tile_kernel(i:cute.Tensor,o:cute.Tensor,block_size: cutlass.Constexpr[int]):  # 编译期常量，tiler 尺寸):
    bidx,_,_ = cute.arch.block_idx()
    tid,_,_ = cute.arch.thread_idx()
    local_tensor = cute.local_tile(i,(block_size,),(bidx,))
    # local_o = cute.local_tile(o,(block_size,),(bidx,))
    if tid==0:
        o[bidx] = cutlass.Float16(bidx * block_size)
        # local_o[0]=cutlass.Float16(bidx * block_size)        # 直接写偏移，不依赖 arange 巧合
        # local_o[0] =  local_tensor[0]

@cute.jit
def ex(i:cute.Tensor,o:cute.Tensor,stream:cuda.CUstream):
    n = i.shape[0]
    nums_block = (n+block_size-1)//block_size
    local_tile_kernel(i,o,block_size).launch(grid=(nums_block,1,1),block=(block_size,1,1),stream = stream)

def demo():
    
    nums_block = (n + block_size - 1) // block_size
    torch_tensor = torch.arange(0,n,dtype=torch.float16,device='cuda')
    i = from_dlpack(torch_tensor).mark_layout_dynamic()
    torch_result = torch.zeros(nums_block,dtype=torch.float16,device='cuda')
    o = from_dlpack(torch_result).mark_layout_dynamic()
    stream = cuda.CUstream(0)
    compiled = cute.compile(ex,i,o,stream)
    compiled(i,o,stream)
    cuda.cuStreamSynchronize(stream)
    print(torch_result.cpu().tolist())
demo()

[0.0, 128.0, 256.0, 384.0, 512.0, 640.0, 768.0, 896.0]


In [3]:
# 边界处理：实现带边界处理的 add_kernel，测试 n=1000（不是 256 的倍数）。
import cutlass
import cutlass.cute as cute   # 官方包是 cutlass，cute 是它的子模块
import torch
from cutlass.cute.runtime import from_dlpack
import cuda.bindings.driver as cuda 
n = 1000
block_size = 128

@cute.kernel
def add_kernel(a:cute.Tensor,
                b:cute.Tensor,
                c:cute.Tensor,
                n,
                block_size : cutlass.Constexpr[int]
                ):
    bidx,_,_ = cute.arch.block_idx()
    tidx,_,_ = cute.arch.thread_idx()
    idx = bidx*block_size + tidx
    
    # block
    a_tile_b = cute.local_tile(a,(block_size,),(bidx,))
    b_tile_b = cute.local_tile(b,(block_size,),(bidx,))
    c_tile_b = cute.local_tile(c,(block_size,),(bidx,))

    # thread
    layout_t = cute.make_layout((block_size,),stride=(1,))#,(1,))
    a_tile_t = cute.local_partition(a_tile_b,layout_t,tidx)
    b_tile_t = cute.local_partition(b_tile_b,layout_t,tidx)
    c_tile_t = cute.local_partition(c_tile_b,layout_t,tidx)
    if(idx<n):
        c_tile_t[0] = a_tile_t[0] + b_tile_t[0]

# ================= 使用 @cute.jit 包装并启动 =================
@cute.jit
def add_jit(a: cute.Tensor, b: cute.Tensor, c: cute.Tensor, stream: cuda.CUstream):
    # 在 JIT 函数内部获取 n 和计算网格大小
    n_val = a.shape[0]
    nums_block = (n_val + block_size - 1) // block_size
    
    # 调用 kernel 并配置 launch 参数
    add_kernel(a, b, c, n_val, block_size).launch(
        grid=(nums_block, 1, 1),
        block=(block_size, 1, 1),
        stream=stream
    )

# ================= 测试代码 =================
def demo():
    # 生成测试数据 (使用 float32 方便验证)
    a_tensor = torch.randn(n, dtype=torch.float32, device='cuda')
    b_tensor = torch.randn(n, dtype=torch.float32, device='cuda')
    c_tensor = torch.zeros(n, dtype=torch.float32, device='cuda')

    # 转换为 cute 的 Tensor，并标记动态布局
    a_cute = from_dlpack(a_tensor).mark_layout_dynamic()
    b_cute = from_dlpack(b_tensor).mark_layout_dynamic()
    c_cute = from_dlpack(c_tensor).mark_layout_dynamic()

    # 创建流并编译
    stream = cuda.CUstream(0)
    compiled = cute.compile(add_jit, a_cute, b_cute, c_cute, stream)
    
    # 运行编译后的函数
    compiled(a_cute, b_cute, c_cute, stream)
    
    # 同步流
    cuda.cuStreamSynchronize(stream)

    # 对比 PyTorch 的加法结果
    ref = a_tensor + b_tensor
    is_correct = torch.allclose(c_tensor, ref, atol=1e-5)

    print(f"测试 n={n} (block_size={block_size}) 的结果: {'通过' if is_correct else '失败'}")
    if not is_correct:
        print("PyTorch ref:", ref[:10])
        print("CUTLASS out:", c_tensor[:10])

demo()


测试 n=1000 (block_size=128) 的结果: 通过


In [4]:
import cutlass
import cutlass.cute as cute
import torch
from cutlass.cute.runtime import from_dlpack
import cuda.bindings.driver as cuda


@cute.kernel
def inspect_kernel(a: cute.Tensor, n: cutlass.Int32):
    tidx, _, _ = cute.arch.thread_idx()

    # ========== 情况 A：1D 分区器 (32,)，launch 64 线程 ==========
    # 演示三件事：ceil 除法、越界槽位、索引回绕
    part_a = cute.make_layout((32,))
    v1 = cute.local_partition(a, part_a, tidx)
    # 探针：identity tensor 与 a 同构、同方式切分，其元素值就是逻辑下标
    c1 = cute.local_partition(cute.make_identity_tensor((n,)), part_a, tidx)
    for e in cutlass.range(cute.size(v1)):          # 逐槽打印，不写死槽位数
        cute.printf("A: tid=%d, elems=%d, slot%d -> a[%d]",
                    tidx, cute.size(v1), e, c1[e])

    # ========== 情况 B：reshape 成 2D 视图 (n/2, 2):(2,1) 再用 1D 分区器 ==========
    # 每线程拿"连续"2 个元素（向量化友好）；探针也必须是 2D
    n2 = n // 2
    part_b = cute.make_layout((64,))
    a2 = cute.make_tensor(a.iterator,
                          cute.make_layout((n2, 2), stride=(2, 1)))
    v2 = cute.local_partition(a2, part_b, tidx)
    c2 = cute.local_partition(cute.make_identity_tensor((n2, 2)), part_b, tidx)
    for e in cutlass.range(cute.size(v2)):
        cute.printf("B: tid=%d, elems=%d, slot%d -> a[%d]  (row=%d, col=%d)",
                    tidx, cute.size(v2), e,
                    c2[e][0] * 2 + c2[e][1],        # 换算回 1D 线性下标
                    c2[e][0], c2[e][1])


@cute.jit
def inspect(a: cute.Tensor, n: cutlass.Int32, stream: cuda.CUstream):
    inspect_kernel(a, n).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)


def demo():
    n = 70
    tensor_ = torch.arange(0, n, device='cuda', dtype=torch.float16)
    a = from_dlpack(tensor_).mark_layout_dynamic()

    stream = cuda.CUstream(0)
    compiled = cute.compile(inspect, a, n, stream)
    compiled(a, n, stream)
    torch.cuda.synchronize()


demo()


A: tid=0, elems=3, slot0 -> a[0]
A: tid=1, elems=3, slot0 -> a[1]
A: tid=2, elems=3, slot0 -> a[2]
A: tid=3, elems=3, slot0 -> a[3]
A: tid=4, elems=3, slot0 -> a[4]
A: tid=5, elems=3, slot0 -> a[5]
A: tid=6, elems=3, slot0 -> a[6]
A: tid=7, elems=3, slot0 -> a[7]
A: tid=8, elems=3, slot0 -> a[8]
A: tid=9, elems=3, slot0 -> a[9]
A: tid=10, elems=3, slot0 -> a[10]
A: tid=11, elems=3, slot0 -> a[11]
A: tid=12, elems=3, slot0 -> a[12]
A: tid=13, elems=3, slot0 -> a[13]
A: tid=14, elems=3, slot0 -> a[14]
A: tid=15, elems=3, slot0 -> a[15]
A: tid=16, elems=3, slot0 -> a[16]
A: tid=17, elems=3, slot0 -> a[17]
A: tid=18, elems=3, slot0 -> a[18]
A: tid=19, elems=3, slot0 -> a[19]
A: tid=20, elems=3, slot0 -> a[20]
A: tid=21, elems=3, slot0 -> a[21]
A: tid=22, elems=3, slot0 -> a[22]
A: tid=23, elems=3, slot0 -> a[23]
A: tid=24, elems=3, slot0 -> a[24]
A: tid=25, elems=3, slot0 -> a[25]
A: tid=26, elems=3, slot0 -> a[26]
A: tid=27, elems=3, slot0 -> a[27]
A: tid=28, elems=3, slot0 -> a[28]
A: ti